# GitHub API Data Pipeline

Rancel Hernandez

This notebook orchestrates a GitHub REST API ingestion pipeline that:
- retrieves users, repositories, commits, and issues
- loads raw data into PostgreSQL staging tables
- transforms data using dbt
- supports incremental ingestion

In [1]:
# imports

# db adapter
import psycopg2

# load data into the db tables
from src.db_helpers import (
    table_insertion
)

# import pipeline variables
from src.config import (
    pipeline_config
)

# main pipeline and function to retreive additional data for existing users
from src.pipeline import (
    run_pipeline,
    get_repos_data,
    get_commits_data,
    get_issues_data
)

# retrieve most recent ingested user
from src.db_helpers import (
    retrieve_last_user_id
)

# print total API calls
from src.api_helpers import (
    print_api_call_count
)

In [2]:
# unpack for db connection
DB_NAME, PASSWORD = pipeline_config["DB_NAME"], pipeline_config["PASSWORD"]

# get most recent user id to get data from after that point
most_recent_user_id = retrieve_last_user_id(DB_NAME, PASSWORD)

# pass to config
pipeline_config["most_recent_user_id"] = most_recent_user_id

# ensure all variables exist
for key, item in pipeline_config.items():
    print(f"{key} exists: {item is not None}")

TOKEN exists: True
DB_NAME exists: True
PASSWORD exists: True
headers exists: True
rename_users_map exists: True
rename_repos_map exists: True
rename_commits_map exists: True
rename_issues_map exists: True
most_recent_user_id exists: True


In [3]:
# fetch data
all_users_data, all_repos_data, all_commits_data, all_issues_data = run_pipeline(**pipeline_config)

# data size
print("Number of users:", len(all_users_data))
print("Number of repos:",len(all_repos_data))
print("Number of commits:", len(all_commits_data))
print("Number of issues:", len(all_issues_data))

Number of users: 60
Number of repos: 530
Number of commits: 7643
Number of issues: 423


In [4]:
# connect to postgres DB
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# insert data into staging tables
table_insertion(conn, 'users', all_users_data)
table_insertion(conn, 'repos', all_repos_data)
table_insertion(conn, 'commits', all_commits_data)
table_insertion(conn, 'issues', all_issues_data)

# get current calls to subtract from next print call
main_pipeline_calls = print_api_call_count()

# close connection
conn.close()

Total API calls: 920


In [5]:
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# get additional repos/commits/issues for existing users
repos_data, commits_data, issues_data = get_repos_data(5, conn, **pipeline_config)

# insert data into db staging tables
table_insertion(conn, 'repos', repos_data)
table_insertion(conn, 'commits', commits_data)
table_insertion(conn, 'issues', issues_data)

# data size
print("Number of repos:",len(repos_data))
print("Number of commits:", len(commits_data))
print("Number of issues:", len(issues_data))

# print api call rate
repo_calls = print_api_call_count()

conn.close()

Number of repos: 244
Number of commits: 3659
Number of issues: 155
Total API calls: 423


In [6]:
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# ingest more commits for existing repos
commits_data = get_commits_data(3, conn, **pipeline_config)

# insert into staging tables
table_insertion(conn, 'commits', commits_data)
print("Number of commits:", len(commits_data))

# print api call rate
commit_calls = print_api_call_count()

conn.close()

Number of commits: 1337
Total API calls: 1101


In [7]:
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

issues_data = get_issues_data(10, conn, **pipeline_config)

# insert into staging tables
table_insertion(conn, 'issues', issues_data)
print("Number of issues:", len(issues_data))

# print api call rate
issue_calls = print_api_call_count()

conn.close()

Number of issues: 58
Total API calls: 466


In [8]:
print("Total Notebook API Calls:", main_pipeline_calls + repo_calls + commit_calls + issue_calls)

Total Notebook API Calls: 2910
